# SimpleNN Distance Training

This notebook trains SimpleNN models on individual distances (d=3, 5, 7, 9, 11) using the best hyperparameters from tuning.

**Workflow:**
1. Load best hyperparameters from tuning results
2. For each distance d ∈ {3, 5, 7, 9, 11}:
   - Generate/load flat dataset (10⁶ samples)
   - Train SimpleNN model (50 epochs)
   - Save model and results
3. Models are saved for evaluation in testing.ipynb

## Imports

In [1]:
# Install dependencies (uncomment if needed)
!pip install stim pymatching numpy matplotlib torch tqdm networkx pandas
!pip install torch_geometric

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import sys
import json
import random
import time
import gc
from pathlib import Path
from datetime import datetime

# Detect if running in Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/Research/QEC/quantum-error-correction/quantum-error-correction/code')
else:
    BASE_PATH = Path('../..')  # code/nn/training -> code/

sys.path.insert(0, str(BASE_PATH))

import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm

# Import from benchmark_models.py
from benchmark_models import (
    SurfaceCodeSampler,
    SimpleNNModel,
    SimpleNN,
    FlatDatasetCache,
)

# Set up paths
TRAINING_DIR = BASE_PATH / "nn" / "training"
TRAINING_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = TRAINING_DIR / "results" / "revised_training"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR = TRAINING_DIR / "models" / "revised_training"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR = TRAINING_DIR / "plots" / "revised_training"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

print(f"\nPaths:")
print(f"  BASE_PATH: {BASE_PATH}")
print(f"  TRAINING_DIR: {TRAINING_DIR}")
print(f"  RESULTS_DIR: {RESULTS_DIR}")
print(f"  MODELS_DIR: {MODELS_DIR}")

C:\Users\Bill\AppData\Roaming\Python\Python313\site-packages\torch\cuda\__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
C:\Users\Bill\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
Using device: cuda
GPU: NVIDIA GeForce RTX 5070 Ti

Paths:
  BASE_PATH: ..\..
  TRAINING_DIR: ..\..\nn\training
  RESULTS_DIR: ..\..\nn\training\results\revised_training
  MODELS_DIR: ..\..\nn\training\models\revised_training


## Configuration

In [3]:
# =============================================================================
# BEST HYPERPARAMETERS (loaded from tuning results)
# =============================================================================

# Load tuning results and find the best configuration
TUNING_RESULTS_PATH = BASE_PATH / "nn" / "tuning" / "results" / "results.json"

with open(TUNING_RESULTS_PATH, 'r') as f:
    tuning_results = json.load(f)

# Filter to completed results and find best by test accuracy
completed = [r for r in tuning_results if r.get('status') == 'completed']
best_result = max(completed, key=lambda x: x['test_accuracy'])
best_config = best_result['config']

# Extract hyperparameters
BEST_HYPERPARAMS = {
    'hidden_dims': tuple(best_config['hidden_dims']),
    'learning_rate': best_config['learning_rate'],
    'batch_size': best_config['batch_size'],
    'dropout': best_config['dropout'],
}

print(f"Loaded best hyperparameters from: {TUNING_RESULTS_PATH}")
print(f"Best config ID: {best_result['config_id']}")
print(f"Tuning performance: val_acc={best_result['val_accuracy']*100:.2f}%, test_acc={best_result['test_accuracy']*100:.2f}%")
print(f"\nHyperparameters:")
for k, v in BEST_HYPERPARAMS.items():
    print(f"  {k}: {v}")

Loaded best hyperparameters from: ..\..\nn\tuning\results\results.json
Best config ID: 33
Tuning performance: val_acc=87.69%, test_acc=87.76%

Hyperparameters:
  hidden_dims: (512, 1024, 2048)
  learning_rate: 0.0003
  batch_size: 64
  dropout: 0.1


In [ ]:
# =============================================================================
# TRAINING CONFIGURATION
# =============================================================================

# Distances to train (one model per distance)
DISTANCES = [3, 5, 7, 9, 11, 13] # 

# Total samples defining the cached eval dataset size (10^6)
TOTAL_SAMPLES = 1_000_000

# Training parameters
EPOCHS = 50

# Random seed for reproducibility
SEED = 42

# Rolling chunk configuration
# Dataset is split into NUM_CHUNKS chunks, each lasting CHUNK_LIFETIME epochs
# After each epoch, the oldest chunk is discarded and a new one is generated
NUM_CHUNKS = 4              # Dataset split into 4 chunks (25% each)
CHUNK_LIFETIME = 4          # Each chunk lasts 4 epochs before being discarded
VALIDATE_EVERY = 1          # run validation every epoch (after each chunk rotation)

# Error rate distribution for generated TRAINING data
P_VALUES = [0.001, 0.003, 0.005, 0.007]
P_WEIGHTS = [0.25, 0.25, 0.25, 0.25]

# Train/test split (test is taken from cached baseline arrays)
TRAIN_RATIO = 0.8
TEST_RATIO = 0.2

# Validation set: 10k samples generated with independent seed
VAL_SAMPLES = 10_000
VAL_SEED = SEED + 999999    # Independent seed for validation data


# Total training samples (all chunks combined)
TRAIN_SAMPLES_TOTAL = int(TOTAL_SAMPLES * TRAIN_RATIO)

# Samples per chunk (25% of training data)
CHUNK_SIZE = TRAIN_SAMPLES_TOTAL // NUM_CHUNKS

print(f"Training Configuration:")
print(f"  Distances: {DISTANCES}")
print(f"  Cached eval samples per distance: {TOTAL_SAMPLES:,}")
print(f"  Training samples total: {TRAIN_SAMPLES_TOTAL:,}")
print(f"  Epochs: {EPOCHS}")
print(f"Rolling Chunk Configuration:")
print(f"  Number of chunks: {NUM_CHUNKS}")
print(f"  Chunk size: {CHUNK_SIZE:,} samples ({100/NUM_CHUNKS:.0f}% of training data)")
print(f"  Chunk lifetime: {CHUNK_LIFETIME} epochs")
print(f"  Validate every: {VALIDATE_EVERY} epoch(s)")
print(f"  Validation samples: {VAL_SAMPLES:,} (independent seed: {VAL_SEED})")
print(f"  Batch size: {BEST_HYPERPARAMS['batch_size']}")
print(f"  P values: {P_VALUES}")
print(f"  P weights: {P_WEIGHTS}")

Training Configuration:
  Distances: [13]
  Cached eval samples per distance: 1,000,000
  Training samples total: 800,000
  Epochs: 50
Rolling Chunk Configuration:
  Number of chunks: 4
  Chunk size: 200,000 samples (25% of training data)
  Chunk lifetime: 4 epochs
  Validate every: 1 epoch(s)
  Validation samples: 10,000 (independent seed: 1000041)
  Batch size: 64
  P values: [0.001, 0.003, 0.005, 0.007]
  P weights: [0.25, 0.25, 0.25, 0.25]


In [6]:
# =============================================================================
# SELECT WHICH DISTANCES TO TRAIN
# =============================================================================

# Set to None to train all distances, or specify a list
# Examples:
#   DISTANCES_TO_TRAIN = None  # Train all
#   DISTANCES_TO_TRAIN = [3]   # Train only d=3
#   DISTANCES_TO_TRAIN = [9, 11]  # Train only d=9 and d=11

DISTANCES_TO_TRAIN = None  # Train all distances

if DISTANCES_TO_TRAIN is None:
    distances_to_train = DISTANCES
else:
    distances_to_train = DISTANCES_TO_TRAIN

print(f"Distances to train: {distances_to_train}")

Distances to train: [13]


## Helper Functions

In [ ]:
def get_num_detectors(d: int) -> int:
    """Calculate number of detectors for a given code distance."""
    # For rotated surface code: d rounds of (d^2 - 1) detectors
    # Actually for surface_code:rotated_memory_z with rounds=d
    # Generate a small sample to get the actual number
    import stim
    circuit = stim.Circuit.generated(
        "surface_code:rotated_memory_z",
        rounds=d,
        distance=d,
        after_clifford_depolarization=0.001,
        after_reset_flip_probability=0.001,
        before_measure_flip_probability=0.001,
        before_round_data_depolarization=0.001
    )
    sampler = circuit.compile_detector_sampler()
    detection_events, _ = sampler.sample(1, separate_observables=True)
    return detection_events.shape[1]


def load_or_generate_dataset(d: int, n_samples: int):
    """Load dataset for a distance strictly from baseline files.

    This function ONLY loads datasets named `d{d}_baseline_array.pt` or
    `d{d}_baseline_array.json` from `code/nn_datasets/`. Any cache files
    ending with `_1M` are ignored per user request. If no baseline
    file is present, a FileNotFoundError is raised.
    """

    flat_dir = BASE_PATH / "nn_datasets"
    pt_path = flat_dir / f"d{d}_baseline_array.pt"
    json_path = flat_dir / f"d{d}_baseline_array.json"

    # Prefer the .pt baseline file
    if pt_path.exists():
        data = torch.load(pt_path, map_location=device)

        # Accept common serialized formats: dict or (detections, labels)
        detections = None
        labels = None
        if isinstance(data, dict):
            # Check keys explicitly to avoid ambiguous truth-value of tensors
            if 'detections' in data:
                detections = data['detections']
            elif 'X' in data:
                detections = data['X']
            elif 'inputs' in data:
                detections = data['inputs']

            if 'labels' in data:
                labels = data['labels']
            elif 'y' in data:
                labels = data['y']
            elif 'targets' in data:
                labels = data['targets']

            if detections is None or labels is None:
                raise ValueError(f"Unexpected .pt format in {pt_path}; expected dict with 'detections' and 'labels'.")

        elif isinstance(data, (list, tuple)) and len(data) >= 2:
            detections, labels = data[0], data[1]
        else:
            raise ValueError(f"Could not parse {pt_path} — expected (detections, labels) or dict.")

        # Convert to tensors on the requested device
        detections = torch.as_tensor(detections, device=device)
        labels = torch.as_tensor(labels, device=device)

        # Ensure float dtype to avoid BCELoss dtype errors
        if not torch.is_floating_point(detections):
            detections = detections.float()
        if not torch.is_floating_point(labels):
            labels = labels.float()

        if len(labels) < n_samples:
            raise FileNotFoundError(f"Baseline file {pt_path} contains only {len(labels)} samples, needed {n_samples}.")

        return detections[:n_samples], labels[:n_samples]

    # Fallback to JSON baseline if present
    if json_path.exists():
        with open(json_path, 'r') as f:
            obj = json.load(f)

        detections = obj.get('detections') if 'detections' in obj else obj.get('X')
        labels = obj.get('labels') if 'labels' in obj else obj.get('y')
        if detections is None or labels is None:
            raise ValueError(f"Unexpected JSON format in {json_path}; expected keys 'detections' and 'labels'.")

        detections = torch.tensor(detections, device=device)
        labels = torch.tensor(labels, device=device)

        # Ensure float dtype
        if not torch.is_floating_point(detections):
            detections = detections.float()
        if not torch.is_floating_point(labels):
            labels = labels.float()

        if len(labels) < n_samples:
            raise FileNotFoundError(f"Baseline file {json_path} contains only {len(labels)} samples, needed {n_samples}.")

        return detections[:n_samples], labels[:n_samples]

    # If no baseline exists, do NOT use _1M caches; raise per user instruction
    raise FileNotFoundError(
        f"No baseline dataset found for d{d} in {flat_dir}.\n"
        "Datasets ending with '_1M' will be ignored. Please provide "
        f"d{d}_baseline_array.pt or d{d}_baseline_array.json in {flat_dir}."
    )


def split_dataset(detections, labels, train_ratio=0.8, val_ratio=0.1, seed=42):
    """Split dataset into train/val/test sets."""
    n = len(labels)

    # Shuffle with fixed seed
    torch.manual_seed(seed)
    perm = torch.randperm(n, device=detections.device)
    detections = detections[perm]
    labels = labels[perm]

    # Split
    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))

    train_det, train_lab = detections[:train_end], labels[:train_end]
    val_det, val_lab = detections[train_end:val_end], labels[train_end:val_end]
    test_det, test_lab = detections[val_end:], labels[val_end:]

    print(f"Split: {len(train_lab):,} train, {len(val_lab):,} val, {len(test_lab):,} test")
    return (train_det, train_lab), (val_det, val_lab), (test_det, test_lab)


def evaluate_model(model, detections, labels, batch_size=256, device=None):
    """Evaluate model accuracy (moves batches to device)."""
    model.model.eval()
    correct = 0
    total = len(labels)

    if device is None:
        device = model.device if hasattr(model, 'device') else (detections.device if hasattr(detections, 'device') else None)

    with torch.no_grad():
        for i in range(0, total, batch_size):
            X = detections[i:i+batch_size]
            y = labels[i:i+batch_size]
            if device is not None:
                X = X.to(device)
                y = y.to(device)
            pred = model.model(X)
            correct += ((pred.squeeze() > 0.5).float() == y).sum().item()

    return correct / total if total > 0 else 0.0


def get_completed_distances():
    """Get set of already-completed distances from results file."""
    results_path = RESULTS_DIR / "results.json"
    if not results_path.exists():
        return set()
    with open(results_path, 'r') as f:
        results = json.load(f)
    return {r['distance'] for r in results if r.get('status') == 'completed'}


def make_json_serializable(obj):
    """Recursively convert tensors and numpy arrays to JSON-serializable types."""
    if isinstance(obj, dict):
        return {k: make_json_serializable(v) for k, v in obj.items() 
                if not k.startswith('_')}  # Skip private keys like _optimizer_state_dict
    elif isinstance(obj, (list, tuple)):
        return [make_json_serializable(v) for v in obj]
    elif isinstance(obj, torch.Tensor):
        if obj.numel() == 1:
            return obj.item()
        return obj.tolist()
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, (np.integer, np.floating)):
        return obj.item()
    elif isinstance(obj, float) and (obj != obj):  # NaN check
        return None
    return obj


def save_result(result):
    """Append a result to the results file."""
    results_path = RESULTS_DIR / "results.json"

    if results_path.exists():
        with open(results_path, 'r') as f:
            results = json.load(f)
    else:
        results = []

    # Convert tensors and numpy types to JSON-serializable types
    result_clean = make_json_serializable(result)
    results.append(result_clean)

    with open(results_path, 'w') as f:
        json.dump(results, f, indent=2)

    return len(results)

In [ ]:
# =============================================================================
# ROLLING CHUNK DATA GENERATION FOR SimpleNN
# =============================================================================

def _mix_seed(base: int, d_val: int, chunk_idx: int) -> int:
    """Deterministic seed mixer (uint32-ish)."""
    return int((base * 1000003 + d_val * 1009 + chunk_idx * 9176) % (2**32 - 1))


def generate_val_data_single(d: int) -> tuple:
    """Generate validation data with independent seed (completely separate from training chunks).
    
    Uses VAL_SEED to ensure validation data is unrelated to training chunks.
    
    Returns:
        Tuple of (detections, labels) tensors on CPU
    """
    print(f"Generating {VAL_SAMPLES:,} validation samples for d={d} with independent seed {VAL_SEED}...")
    
    # Use independent validation seed (add d for uniqueness per distance)
    val_seed = VAL_SEED + d
    random.seed(val_seed)
    np.random.seed(val_seed % (2**32 - 1))
    torch.manual_seed(val_seed)
    
    sampler = SurfaceCodeSampler(p=P_VALUES[0], device=torch.device('cpu'))
    det, lab = sampler.sample(
        d=d,
        num_samples=VAL_SAMPLES,
        p_values=P_VALUES,
        p_weights=P_WEIGHTS,
    )
    
    print(f"Validation data generated for d={d}: {VAL_SAMPLES:,} samples")
    return det.cpu().float(), lab.cpu().float()


def generate_chunk_single(d: int, chunk_size: int, chunk_idx: int) -> tuple:
    """Generate a single chunk of training data for a single distance.
    
    Args:
        d: Code distance
        chunk_size: Number of samples in this chunk
        chunk_idx: Chunk index for deterministic seeding
        
    Returns:
        Tuple of (detections, labels) tensors on CPU
    """
    print(f"  [chunk {chunk_idx}] Generating {chunk_size:,} samples for d={d}...")

    seed_i = _mix_seed(SEED, d, chunk_idx)
    random.seed(seed_i)
    np.random.seed(seed_i % (2**32 - 1))
    torch.manual_seed(seed_i)

    sampler = SurfaceCodeSampler(p=P_VALUES[0], device=torch.device('cpu'))
    det, lab = sampler.sample(
        d=d,
        num_samples=chunk_size,
        p_values=P_VALUES,
        p_weights=P_WEIGHTS,
    )
    return det.cpu(), lab.cpu()


def get_active_chunk_indices(epoch: int, num_chunks: int = NUM_CHUNKS) -> list:
    """Return list of chunk indices that should be active at given epoch."""
    return list(range(epoch, epoch + num_chunks))


def initialize_chunks_single(d: int, start_epoch: int, 
                             chunk_size: int = CHUNK_SIZE, num_chunks: int = NUM_CHUNKS) -> dict:
    """Initialize all chunks needed for the starting epoch for a single distance.
    
    Returns:
        Dict mapping chunk_idx -> (detections, labels) tuple
    """
    print(f"\n[Rolling chunks] Initializing {num_chunks} chunks for d={d}, epoch {start_epoch}...")
    chunks = {}
    for chunk_idx in get_active_chunk_indices(start_epoch, num_chunks):
        chunks[chunk_idx] = generate_chunk_single(d, chunk_size, chunk_idx)
    total_samples = sum(len(c[1]) for c in chunks.values())
    print(f"[Rolling chunks] Initialized: {len(chunks)} chunks, {total_samples:,} total samples")
    return chunks


def roll_chunks_single(chunks: dict, d: int, new_epoch: int,
                       chunk_size: int = CHUNK_SIZE, num_chunks: int = NUM_CHUNKS) -> dict:
    """Roll the chunk window: discard oldest chunk, generate new chunk."""
    import gc
    
    oldest_idx = new_epoch - num_chunks
    if oldest_idx in chunks:
        print(f"\n[Rolling chunks] Epoch {new_epoch}: discarding chunk {oldest_idx}, ", end="")
        del chunks[oldest_idx]
        gc.collect()
    
    new_idx = new_epoch + num_chunks - 1
    if new_idx not in chunks:
        print(f"generating chunk {new_idx}...")
        chunks[new_idx] = generate_chunk_single(d, chunk_size, new_idx)
    
    return chunks


def combine_chunks_flat(chunks: dict) -> tuple:
    """Combine all chunks into single (detections, labels) tensors."""
    det_list = []
    lab_list = []
    for chunk_idx in sorted(chunks.keys()):
        det, lab = chunks[chunk_idx]
        det_list.append(det)
        lab_list.append(lab)
    
    combined_det = torch.cat(det_list, dim=0)
    combined_lab = torch.cat(lab_list, dim=0)
    
    # Shuffle the combined data
    perm = torch.randperm(len(combined_lab))
    return combined_det[perm], combined_lab[perm]


print("Rolling chunk functions defined.")

In [ ]:
# =============================================================================
# TRAINING WITH LOGGING (adapted for SimpleNN with flat tensor data)
# =============================================================================

def train_with_logging(
    model,
    val_det,
    val_lab,
    epochs,
    batch_size,
    lr,
    device,
    # Training data refresh
    generate_train_data_fn,
    refresh_every: int = 10,
    validate_every: int = 10,
    # Gradient clipping (relaxed since root causes are fixed)
    max_grad_norm: float = 1.0,
    # Learning rate warmup
    warmup_epochs: int = 5,
    # Checkpoint parameters (optional)
    experiment_name: str = None,
    distance: int = None,
    hyperparams: dict = None,
    training_config: dict = None,
    models_dir: Path = None,
    # Resumption parameters
    start_epoch: int = 0,
    initial_metrics: dict = None,
    optimizer_state_dict: dict = None,
    verbose: bool = True
):
    """Train SimpleNN with per-epoch logging + training-data refresh.

    Behavior:
    - Training data is regenerated every `refresh_every` epochs via `generate_train_data_fn(block_idx)`.
    - Validation is computed every `validate_every` epochs and at the final epoch.
    - Optimizer state is preserved across refresh blocks.

    Args:
        model: SimpleNN model wrapper
        val_det/val_lab: Fixed validation set (typically loaded from cached baseline arrays)
        epochs: Total number of epochs
        batch_size: Batch size
        lr: Learning rate
        device: Torch device
        generate_train_data_fn: Callable(block_idx:int) -> (train_det_cpu, train_lab_cpu)
        refresh_every: Regenerate training set every N epochs
        validate_every: Evaluate validation every N epochs
        start_epoch: Resume start epoch (0-indexed)
        initial_metrics: Pre-existing epoch metrics if resuming
        optimizer_state_dict: Optimizer state to restore if resuming

    Returns:
        Dict with per-epoch metrics
    """
    # Ensure model is initialized
    if model.model is None:
        raise ValueError("Model must be initialized before training (in_channels must be known).")

    model.model.train()
    optimizer = torch.optim.AdamW(model.model.parameters(), lr=lr, weight_decay=0.01)
    loss_fn = torch.nn.BCELoss()

    # Restore optimizer state if resuming
    if optimizer_state_dict is not None:
        optimizer.load_state_dict(optimizer_state_dict)
        if verbose:
            print("Restored optimizer state from checkpoint")

    # Initialize or restore logging containers
    if initial_metrics is not None:
        epoch_metrics = initial_metrics.copy()
        if verbose:
            print(f"Restored {len(epoch_metrics['epoch'])} epochs of metrics from checkpoint")
    else:
        epoch_metrics = {
            'epoch': [],
            'train_loss': [],
            'train_accuracy': [],
            'val_accuracy': [],
            'epoch_time_seconds': [],
            'learning_rate': [],
            'gpu_memory_mb': [],
            'train_block_idx': [],
            'max_grad_norm': [],  # Track gradient norms for debugging
        }

    # Check if checkpointing is enabled
    checkpoint_enabled = all([
        experiment_name is not None,
        distance is not None,
        hyperparams is not None,
        training_config is not None,
        models_dir is not None
    ])

    if verbose:
        print(f"\n{'='*70}")
        print(f"Training: {model.nickname}")
        print(f"{'='*70}")
        print(f"Epochs: {start_epoch + 1} to {epochs} | Batch size: {batch_size} | LR: {lr}")
        print(f"Validation samples: {len(val_lab):,}")
        print(f"Refresh every: {refresh_every} epochs | Validate every: {validate_every} epochs")
        print(f"Checkpointing: {'Enabled' if checkpoint_enabled else 'Disabled'}")
        print(f"{'='*70}\n")

    total_start_time = time.time()

    # Initial training block
    current_block_idx = start_epoch // refresh_every
    train_det, train_lab = generate_train_data_fn(current_block_idx)

    for epoch in range(start_epoch, epochs):
        epoch_start_time = time.time()

        # Learning rate warmup: ramp from 10% to 100% over warmup_epochs
        if epoch < warmup_epochs:
            current_lr = lr * (0.1 + 0.9 * epoch / warmup_epochs)
            for param_group in optimizer.param_groups:
                param_group['lr'] = current_lr
        else:
            current_lr = lr
            for param_group in optimizer.param_groups:
                param_group['lr'] = lr

        # Refresh training data at boundaries
        desired_block_idx = epoch // refresh_every
        if desired_block_idx != current_block_idx:
            current_block_idx = desired_block_idx
            # Free old data
            del train_det, train_lab
            gc.collect()
            train_det, train_lab = generate_train_data_fn(current_block_idx)

        # Training phase
        model.model.train()

        # Keep training data on CPU; shuffle on CPU
        num_samples = len(train_lab)
        num_batches = (num_samples + batch_size - 1) // batch_size
        perm = torch.randperm(num_samples, device=train_lab.device)
        train_det_shuffled = train_det[perm]
        train_lab_shuffled = train_lab[perm]

        epoch_loss = 0.0
        epoch_correct = 0
        epoch_total = 0
        epoch_max_grad_norm = 0.0  # Track gradient norms

        pbar = tqdm(range(num_batches), desc=f"Epoch {epoch+1}/{epochs}",
                    disable=not verbose, leave=False)

        for batch_idx in pbar:
            start_idx = batch_idx * batch_size
            end_idx = min(start_idx + batch_size, num_samples)

            X = train_det_shuffled[start_idx:end_idx].to(device)
            y = train_lab_shuffled[start_idx:end_idx].view(-1, 1).float().to(device)

            pred = model.model(X)
            loss = loss_fn(pred, y)

            optimizer.zero_grad()
            loss.backward()
            
            # Monitor gradient norm BEFORE clipping
            grad_norm = 0.0
            for p in model.model.parameters():
                if p.grad is not None:
                    grad_norm += p.grad.data.norm(2).item() ** 2
            grad_norm = grad_norm ** 0.5
            epoch_max_grad_norm = max(epoch_max_grad_norm, grad_norm)
            
            if max_grad_norm is not None:
                torch.nn.utils.clip_grad_norm_(model.model.parameters(), max_grad_norm)
            optimizer.step()

            batch_loss = loss.item()
            epoch_loss += batch_loss
            epoch_correct += ((pred > 0.5).float() == y).sum().item()
            epoch_total += y.size(0)

            current_acc = 100.0 * epoch_correct / epoch_total
            pbar.set_postfix({'loss': f'{batch_loss:.4f}', 'acc': f'{current_acc:.1f}%'})

        pbar.close()

        avg_train_loss = epoch_loss / num_batches
        train_accuracy = epoch_correct / epoch_total

        # Validation only at boundaries and at final epoch
        do_validate = ((epoch + 1) % validate_every == 0) or (epoch + 1 == epochs)
        if do_validate:
            val_accuracy = evaluate_model(model, val_det, val_lab, batch_size=batch_size, device=device)
        else:
            val_accuracy = float('nan')

        epoch_time = time.time() - epoch_start_time

        # GPU memory usage
        if torch.cuda.is_available():
            gpu_memory_mb = torch.cuda.max_memory_allocated() / (1024 * 1024)
            torch.cuda.reset_peak_memory_stats()
        else:
            gpu_memory_mb = 0.0

        # Log metrics
        epoch_metrics['epoch'].append(epoch + 1)
        epoch_metrics['train_loss'].append(avg_train_loss)
        epoch_metrics['train_accuracy'].append(train_accuracy)
        epoch_metrics['val_accuracy'].append(val_accuracy)
        epoch_metrics['epoch_time_seconds'].append(epoch_time)
        epoch_metrics['learning_rate'].append(current_lr)
        epoch_metrics['gpu_memory_mb'].append(gpu_memory_mb)
        epoch_metrics['train_block_idx'].append(current_block_idx)
        epoch_metrics['max_grad_norm'].append(epoch_max_grad_norm)

        if verbose:
            val_str = f"{val_accuracy*100:.2f}%" if (val_accuracy == val_accuracy) else "(skipped)"
            # Warn if gradient norm is very high
            grad_warn = " ⚠️GRAD!" if epoch_max_grad_norm > 10.0 else ""
            print(f"Epoch {epoch+1:3d}/{epochs} | "
                  f"Loss: {avg_train_loss:.4f} | "
                  f"Train Acc: {train_accuracy*100:.2f}% | "
                  f"Val Acc: {val_str} | "
                  f"GradNorm: {epoch_max_grad_norm:.1f}{grad_warn} | "
                  f"Time: {epoch_time:.1f}s")

        # Save checkpoint after each epoch (if enabled)
        if checkpoint_enabled:
            is_final = (epoch + 1 == epochs)
            checkpoint = {
                'state_dict': model.model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'current_epoch': epoch + 1,
                'epoch_metrics': epoch_metrics,
                'training_complete': is_final,
                'config': model._config,
                'experiment_name': experiment_name,
                'distance': distance,
                'hyperparams': hyperparams,
                'training_config': training_config,
                'refresh_every': refresh_every,
                'validate_every': validate_every,
                'current_block_idx': current_block_idx,
                'timestamp': datetime.now().isoformat(),
            }
            checkpoint_path = models_dir / f"{experiment_name}.pt"
            torch.save(checkpoint, checkpoint_path)
            if verbose and is_final:
                print("  [Checkpoint saved - training complete]")

    total_time = time.time() - total_start_time

    if verbose:
        print(f"\n{'='*70}")
        print("Training complete!")
        print(f"Total time: {total_time/60:.1f} minutes")
        print(f"Final train loss: {epoch_metrics['train_loss'][-1]:.4f}")
        print(f"Final train accuracy: {epoch_metrics['train_accuracy'][-1]*100:.2f}%")
        # last val may be nan if validate_every doesn't land on final epoch (we force final epoch validate)
        final_val = epoch_metrics['val_accuracy'][-1]
        if final_val == final_val:
            print(f"Final val accuracy: {final_val*100:.2f}%")
        print(f"{'='*70}")

    epoch_metrics['total_training_time_seconds'] = total_time

    # Return optimizer state too (useful for external chaining)
    epoch_metrics['_optimizer_state_dict'] = optimizer.state_dict()

    return epoch_metrics


def save_training_plot(epoch_metrics, distance, test_accuracy, plots_dir):
    """
    Save training curves plot for a specific distance.
    
    Args:
        epoch_metrics: Dict with per-epoch metrics from train_with_logging
        distance: Code distance (for title)
        test_accuracy: Final test accuracy
        plots_dir: Directory to save plot
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    epochs_list = epoch_metrics['epoch']
    
    # Plot 1: Loss curve
    ax1.plot(epochs_list, epoch_metrics['train_loss'], 'b-', linewidth=2, label='Train Loss')
    ax1.set_xlabel('Epoch', fontsize=12)
    ax1.set_ylabel('Loss', fontsize=12)
    ax1.set_title(f'Training Loss - d={distance}', fontsize=14)
    ax1.legend(loc='upper right')
    ax1.grid(True, alpha=0.3)
    ax1.set_xlim(1, len(epochs_list))
    
    # Plot 2: Accuracy curves
    ax2.plot(epochs_list, [acc*100 for acc in epoch_metrics['train_accuracy']], 
             'b-', linewidth=2, label='Train Accuracy')
    ax2.plot(epochs_list, [acc*100 for acc in epoch_metrics['val_accuracy']], 
             'r-', linewidth=2, label='Val Accuracy')
    ax2.axhline(y=test_accuracy*100, color='g', linestyle='--', linewidth=1.5, 
                label=f'Test Accuracy ({test_accuracy*100:.1f}%)')
    ax2.set_xlabel('Epoch', fontsize=12)
    ax2.set_ylabel('Accuracy (%)', fontsize=12)
    ax2.set_title(f'Training/Validation Accuracy - d={distance}', fontsize=14)
    ax2.legend(loc='lower right')
    ax2.grid(True, alpha=0.3)
    ax2.set_xlim(1, len(epochs_list))
    ax2.set_ylim(50, 100)
    
    plt.tight_layout()
    
    plot_path = plots_dir / f"d{distance}_training.png"
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    
    print(f"Training plot saved to: {plot_path}")


print("Training and plotting functions defined.")

In [ ]:
# =============================================================================
# TRAINING WITH ROLLING CHUNKS (replaces train_with_logging for refresh)
# =============================================================================

def train_with_rolling_chunks(
    model,
    d: int,  # Code distance for this specialist model
    val_det,
    val_lab,
    epochs,
    batch_size,
    lr,
    device,
    validate_every: int = 10,
    max_grad_norm: float = 1.0,
    warmup_epochs: int = 5,
    # Checkpoint parameters (optional)
    experiment_name: str = None,
    distance: int = None,
    hyperparams: dict = None,
    training_config: dict = None,
    models_dir: Path = None,
    # Resumption parameters
    start_epoch: int = 0,
    initial_metrics: dict = None,
    optimizer_state_dict: dict = None,
    initial_chunk_indices: list = None,
    verbose: bool = True
):
    """Train SimpleNN with rolling chunk data refresh.
    
    Instead of replacing the entire dataset every N epochs, we maintain
    NUM_CHUNKS chunks where each chunk lasts CHUNK_LIFETIME epochs.
    After each epoch, the oldest chunk is discarded and a new one is generated.
    """
    # Ensure model is initialized
    if model.model is None:
        raise ValueError("Model must be initialized before training (in_channels must be known).")

    model.model.train()
    optimizer = torch.optim.AdamW(model.model.parameters(), lr=lr, weight_decay=0.01)
    loss_fn = torch.nn.BCELoss()

    # Restore optimizer state if resuming
    if optimizer_state_dict is not None:
        optimizer.load_state_dict(optimizer_state_dict)
        if verbose:
            print("Restored optimizer state from checkpoint")

    # Initialize or restore logging containers
    if initial_metrics is not None:
        epoch_metrics = initial_metrics.copy()
        if verbose:
            print(f"Restored {len(epoch_metrics['epoch'])} epochs of metrics from checkpoint")
    else:
        epoch_metrics = {
            'epoch': [],
            'train_loss': [],
            'train_accuracy': [],
            'val_accuracy': [],
            'epoch_time_seconds': [],
            'learning_rate': [],
            'gpu_memory_mb': [],
            'active_chunks': [],
            'max_grad_norm': [],
        }

    # Check if checkpointing is enabled
    checkpoint_enabled = all([
        experiment_name is not None,
        distance is not None,
        hyperparams is not None,
        training_config is not None,
        models_dir is not None
    ])

    if verbose:
        print(f"\n{'='*70}")
        print(f"Training: {model.nickname}")
        print(f"{'='*70}")
        print(f"Epochs: {start_epoch + 1} to {epochs} | Batch size: {batch_size} | LR: {lr}")
        print(f"Validation samples: {len(val_lab):,}")
        print(f"Rolling chunks: {NUM_CHUNKS} chunks x {CHUNK_SIZE:,} samples, {CHUNK_LIFETIME} epoch lifetime")
        print(f"Validate every: {validate_every} epochs")
        print(f"Checkpointing: {'Enabled' if checkpoint_enabled else 'Disabled'}")
        print(f"{'='*70}\n")

    total_start_time = time.time()

    # Initialize chunks for starting epoch
    if initial_chunk_indices is not None and len(initial_chunk_indices) > 0:
        print(f"\n[Rolling chunks] Regenerating chunks from checkpoint: {initial_chunk_indices}")
        chunks = {}
        for chunk_idx in initial_chunk_indices:
            chunks[chunk_idx] = generate_chunk_single(d, CHUNK_SIZE, chunk_idx)
    else:
        chunks = initialize_chunks_single(d, start_epoch)
    
    train_det, train_lab = combine_chunks_flat(chunks)

    for epoch in range(start_epoch, epochs):
        epoch_start_time = time.time()

        # Learning rate warmup
        if epoch < warmup_epochs:
            current_lr = lr * (0.1 + 0.9 * epoch / warmup_epochs)
            for param_group in optimizer.param_groups:
                param_group['lr'] = current_lr
        else:
            current_lr = lr
            for param_group in optimizer.param_groups:
                param_group['lr'] = lr

        # Roll chunks if not the first epoch
        if epoch > start_epoch:
            chunks = roll_chunks_single(chunks, d, epoch)
            train_det, train_lab = combine_chunks_flat(chunks)

        model.model.train()

        num_samples = len(train_lab)
        num_batches = (num_samples + batch_size - 1) // batch_size
        perm = torch.randperm(num_samples)
        train_det_shuffled = train_det[perm]
        train_lab_shuffled = train_lab[perm]

        epoch_loss = 0.0
        epoch_correct = 0
        epoch_total = 0
        epoch_max_grad_norm = 0.0

        active_chunk_list = sorted(chunks.keys())
        pbar = tqdm(range(num_batches), desc=f"Epoch {epoch+1}/{epochs} [chunks {active_chunk_list}]",
                    disable=not verbose, leave=False)

        for batch_idx in pbar:
            start_idx = batch_idx * batch_size
            end_idx = min(start_idx + batch_size, num_samples)

            X = train_det_shuffled[start_idx:end_idx].to(device)
            y = train_lab_shuffled[start_idx:end_idx].view(-1, 1).float().to(device)

            pred = model.model(X)
            loss = loss_fn(pred, y)

            optimizer.zero_grad()
            loss.backward()
            
            # Monitor gradient norm
            grad_norm = 0.0
            for p in model.model.parameters():
                if p.grad is not None:
                    grad_norm += p.grad.data.norm(2).item() ** 2
            grad_norm = grad_norm ** 0.5
            epoch_max_grad_norm = max(epoch_max_grad_norm, grad_norm)
            
            if max_grad_norm is not None:
                torch.nn.utils.clip_grad_norm_(model.model.parameters(), max_grad_norm)
            optimizer.step()

            batch_loss = loss.item()
            epoch_loss += batch_loss
            epoch_correct += ((pred > 0.5).float() == y).sum().item()
            epoch_total += y.size(0)

            current_acc = 100.0 * epoch_correct / epoch_total
            pbar.set_postfix({'loss': f'{batch_loss:.4f}', 'acc': f'{current_acc:.1f}%'})

        pbar.close()

        avg_train_loss = epoch_loss / num_batches
        train_accuracy = epoch_correct / epoch_total

        # Validation only at boundaries and at final epoch
        do_validate = ((epoch + 1) % validate_every == 0) or (epoch + 1 == epochs)
        if do_validate:
            val_accuracy = evaluate_model(model, val_det, val_lab, batch_size=batch_size, device=device)
        else:
            val_accuracy = float('nan')

        epoch_time = time.time() - epoch_start_time

        # GPU memory usage
        if torch.cuda.is_available():
            gpu_memory_mb = torch.cuda.max_memory_allocated() / (1024 * 1024)
            torch.cuda.reset_peak_memory_stats()
        else:
            gpu_memory_mb = 0.0

        # Log metrics
        epoch_metrics['epoch'].append(epoch + 1)
        epoch_metrics['train_loss'].append(avg_train_loss)
        epoch_metrics['train_accuracy'].append(train_accuracy)
        epoch_metrics['val_accuracy'].append(val_accuracy)
        epoch_metrics['epoch_time_seconds'].append(epoch_time)
        epoch_metrics['learning_rate'].append(current_lr)
        epoch_metrics['gpu_memory_mb'].append(gpu_memory_mb)
        epoch_metrics['active_chunks'].append(active_chunk_list)
        epoch_metrics['max_grad_norm'].append(epoch_max_grad_norm)

        if verbose:
            val_str = f"{val_accuracy*100:.2f}%" if (val_accuracy == val_accuracy) else "(skipped)"
            grad_warn = " GRAD!" if epoch_max_grad_norm > 10.0 else ""
            print(f"Epoch {epoch+1:3d}/{epochs} | "
                  f"Chunks: {active_chunk_list} | "
                  f"Loss: {avg_train_loss:.4f} | "
                  f"Train: {train_accuracy*100:.2f}% | "
                  f"Val: {val_str} | "
                  f"GradNorm: {epoch_max_grad_norm:.1f}{grad_warn} | "
                  f"Time: {epoch_time:.1f}s")

        # Save checkpoint after each epoch (if enabled)
        if checkpoint_enabled:
            is_final = (epoch + 1 == epochs)
            checkpoint = {
                'state_dict': model.model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'current_epoch': epoch + 1,
                'epoch_metrics': epoch_metrics,
                'training_complete': is_final,
                'config': model._config,
                'experiment_name': experiment_name,
                'distance': distance,
                'hyperparams': hyperparams,
                'training_config': training_config,
                'validate_every': validate_every,
                'active_chunk_indices': active_chunk_list,
                'timestamp': datetime.now().isoformat(),
            }
            checkpoint_path = models_dir / f"{experiment_name}.pt"
            torch.save(checkpoint, checkpoint_path)
            if verbose and is_final:
                print("  [Checkpoint saved - training complete]")

    total_time = time.time() - total_start_time

    if verbose:
        print(f"\n{'='*70}")
        print("Training complete!")
        print(f"Total time: {total_time/60:.1f} minutes")
        print(f"Final train loss: {epoch_metrics['train_loss'][-1]:.4f}")
        print(f"Final train accuracy: {epoch_metrics['train_accuracy'][-1]*100:.2f}%")
        final_val = epoch_metrics['val_accuracy'][-1]
        if final_val == final_val:
            print(f"Final val accuracy: {final_val*100:.2f}%")
        print(f"{'='*70}")

    epoch_metrics['total_training_time_seconds'] = total_time
    epoch_metrics['_optimizer_state_dict'] = optimizer.state_dict()

    return epoch_metrics


print("Rolling chunks training function defined.")

## Training Loop

Train one SimpleNN model per distance. Each model has a fixed input size matching the number of detectors for that distance.

In [ ]:
# =============================================================================
# MAIN TRAINING LOOP
# =============================================================================

# Check which distances are already completed
completed_distances = get_completed_distances()
remaining_distances = [d for d in distances_to_train if d not in completed_distances]

print(f"Already completed: {sorted(completed_distances)}")
print(f"Remaining: {remaining_distances}")

if len(remaining_distances) == 0:
    print("\nAll distances are already completed!")
else:
    print(f"\n{'='*60}")
    print(f"Starting training - {len(remaining_distances)} distances")
    print(f"{'='*60}")

    for i, d in enumerate(remaining_distances):
        print(f"\n{'='*60}")
        print(f"Distance d={d} ({i+1}/{len(remaining_distances)})")
        print(f"{'='*60}")

        start_time = time.time()

        try:
            # -----------------------------------------------------------------
            # Generate validation set (10k samples with independent seed)
            # -----------------------------------------------------------------
            val_det, val_lab = generate_val_data_single(d)
            
            # -----------------------------------------------------------------
            # Load test set from cached baseline arrays
            # -----------------------------------------------------------------
            eval_det, eval_lab = load_or_generate_dataset(d, TOTAL_SAMPLES)
            (_, _), (_, _), (test_det, test_lab) = split_dataset(
                eval_det, eval_lab, TRAIN_RATIO, TEST_RATIO, SEED
            )
            del eval_det, eval_lab  # Don't need train/val from cache

            # Get number of detectors for this distance
            num_detectors = val_det.shape[1]
            print(f"Number of detectors: {num_detectors}")

            # Initialize model with best hyperparameters
            model = SimpleNN(
                nickname=f"d{d}_specialist",
                in_channels=num_detectors,
                hidden_dims=BEST_HYPERPARAMS['hidden_dims'],
                dropout=BEST_HYPERPARAMS['dropout'],
                device=device,
                base_path=BASE_PATH,
                seed=SEED + d  # Different seed per distance for variety
            )

            # Scale learning rate inversely proportional to input_dim^0.75
            # More aggressive scaling for large distances to prevent gradient explosion
            base_detectors = 24  # d=3 baseline
            lr_scale = (base_detectors / num_detectors) ** 0.75
            scaled_lr = BEST_HYPERPARAMS['learning_rate'] * lr_scale

            print(f"\nTraining with:")
            print(f"  hidden_dims: {BEST_HYPERPARAMS['hidden_dims']}")
            print(f"  base_learning_rate: {BEST_HYPERPARAMS['learning_rate']}")
            print(f"  lr_scale for d={d}: {lr_scale}")
            print(f"  effective_learning_rate: {scaled_lr}")
            print(f"  batch_size: {BEST_HYPERPARAMS['batch_size']}")
            print(f"  dropout: {BEST_HYPERPARAMS['dropout']}")

            # -----------------------------------------------------------------
            # Training data using rolling chunks
            # -----------------------------------------------------------------
            # Train the model with rolling chunk refresh
            epoch_metrics = train_with_rolling_chunks(
                model=model,
                d=d,
                val_det=val_det.cpu(),
                val_lab=val_lab.cpu(),
                epochs=EPOCHS,
                batch_size=BEST_HYPERPARAMS['batch_size'],
                lr=scaled_lr,  # Use distance-scaled learning rate
                device=device,
                validate_every=VALIDATE_EVERY,
                experiment_name=f"d{d}",
                distance=d,
                hyperparams=BEST_HYPERPARAMS,
                training_config={
                    'epochs': EPOCHS,
                    'batch_size': BEST_HYPERPARAMS['batch_size'],
                    'num_chunks': NUM_CHUNKS,
                    'chunk_size': CHUNK_SIZE,
                    'chunk_lifetime': CHUNK_LIFETIME,
                    'validate_every': VALIDATE_EVERY,
                    'train_samples_total': TRAIN_SAMPLES_TOTAL,
                    'val_samples': VAL_SAMPLES,
                    'val_seed': VAL_SEED,
                    'p_values': P_VALUES,
                    'p_weights': P_WEIGHTS,
                },
                models_dir=MODELS_DIR,
                verbose=True
            )

            # Evaluate on test set (once at the end)
            test_accuracy = evaluate_model(model, test_det.cpu(), test_lab.cpu(), batch_size=BEST_HYPERPARAMS['batch_size'], device=device)
            val_accuracy = epoch_metrics['val_accuracy'][-1]

            training_time = time.time() - start_time

            # Save training plot
            save_training_plot(epoch_metrics, d, test_accuracy, PLOTS_DIR)

            # Save model checkpoint
            model_path = MODELS_DIR / f"d{d}_specialist.pt"
            torch.save({
                'state_dict': model.model.state_dict(),
                'distance': d,
                'num_detectors': num_detectors,
                'hyperparams': BEST_HYPERPARAMS,
                'effective_learning_rate': scaled_lr,
                'lr_scale': lr_scale,
                'val_accuracy': val_accuracy,
                'test_accuracy': test_accuracy,
                'epoch_metrics': epoch_metrics,
            }, model_path)

            # Record result
            result = {
                'distance': d,
                'num_detectors': num_detectors,
                'hyperparams': {
                    'hidden_dims': list(BEST_HYPERPARAMS['hidden_dims']),
                    'base_learning_rate': BEST_HYPERPARAMS['learning_rate'],
                    'lr_scale': lr_scale,
                    'effective_learning_rate': scaled_lr,
                    'batch_size': BEST_HYPERPARAMS['batch_size'],
                    'dropout': BEST_HYPERPARAMS['dropout'],
                },
                'status': 'completed',
                'val_accuracy': val_accuracy,
                'test_accuracy': test_accuracy,
                'final_loss': epoch_metrics['train_loss'][-1],
                'training_time_seconds': training_time,
                'total_samples': TOTAL_SAMPLES,
                'epochs': EPOCHS,
                'model_path': str(model_path),
                'epoch_metrics': epoch_metrics,
                'timestamp': datetime.now().isoformat(),
            }

            n_saved = save_result(result)

            print(f"\nDistance d={d} COMPLETED")
            print(f"  Val accuracy: {val_accuracy:.4f}")
            print(f"  Test accuracy: {test_accuracy:.4f}")
            print(f"  Final loss: {epoch_metrics['train_loss'][-1]:.4f}")
            print(f"  Training time: {training_time:.1f}s ({training_time/60:.1f} min)")
            print(f"  Model saved: {model_path}")

        except Exception as e:
            print(f"\nDistance d={d} FAILED: {e}")
            import traceback
            traceback.print_exc()
            result = {
                'distance': d,
                'status': 'failed',
                'error': str(e),
                'timestamp': datetime.now().isoformat(),
            }
            save_result(result)

        finally:
            # Clean up GPU memory
            if 'model' in dir():
                del model
            if 'detections' in dir():
                del detections, labels
            if 'train_det' in dir():
                del train_det, train_lab, val_det, val_lab, test_det, test_lab
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    print(f"\n{'='*60}")
    print("ALL TRAINING COMPLETE")
    print(f"{'='*60}")

Already completed: []
Remaining: [3, 5, 7, 9, 11]

Starting training - 5 distances

Distance d=3 (1/5)
Split: 800,000 train, 100,000 val, 100,000 test
Number of detectors: 24
SimpleNN initialized: SimpleNN(nickname='d3_specialist', in_channels=24, hidden_dims=(512, 1024, 2048), dropout=0.1)

Training with:
  hidden_dims: (512, 1024, 2048)
  learning_rate: 0.0003
  batch_size: 64
  dropout: 0.1

Training: d3_specialist
Epochs: 10 | Batch size: 64 | LR: 0.0003
Training samples: 800,000
Validation samples: 100,000



Traceback (most recent call last):                   
  File "C:\Users\Win10\AppData\Local\Temp\ipykernel_18204\684687717.py", line 57, in <module>
    epoch_metrics = train_with_logging(
                    ^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Win10\AppData\Local\Temp\ipykernel_18204\3078673785.py", line 210, in train_with_logging
    loss = loss_fn(pred, y)
           ^^^^^^^^^^^^^^^^
  File "c:\Users\Win10\anaconda3\envs\gtx1080-eng\Lib\site-packages\torch\nn\modules\module.py", line 1518, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Win10\anaconda3\envs\gtx1080-eng\Lib\site-packages\torch\nn\modules\module.py", line 1527, in _call_impl
    return forward_call(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Win10\anaconda3\envs\gtx1080-eng\Lib\site-packages\torch\nn\modules\loss.py", line 618, in forward
    return F.binary_cross_entropy(input, target, weight=self.weight, red


Distance d=3 FAILED: Found dtype Long but expected Float

Distance d=5 (2/5)
Split: 800,000 train, 100,000 val, 100,000 test
Number of detectors: 120
SimpleNN initialized: SimpleNN(nickname='d5_specialist', in_channels=120, hidden_dims=(512, 1024, 2048), dropout=0.1)

Training with:
  hidden_dims: (512, 1024, 2048)
  learning_rate: 0.0003
  batch_size: 64
  dropout: 0.1

Training: d5_specialist
Epochs: 10 | Batch size: 64 | LR: 0.0003
Training samples: 800,000
Validation samples: 100,000




Distance d=5 FAILED: Found dtype Long but expected Float


Traceback (most recent call last):
  File "C:\Users\Win10\AppData\Local\Temp\ipykernel_18204\684687717.py", line 57, in <module>
    epoch_metrics = train_with_logging(
                    ^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Win10\AppData\Local\Temp\ipykernel_18204\3078673785.py", line 210, in train_with_logging
    loss = loss_fn(pred, y)
           ^^^^^^^^^^^^^^^^
  File "c:\Users\Win10\anaconda3\envs\gtx1080-eng\Lib\site-packages\torch\nn\modules\module.py", line 1518, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Win10\anaconda3\envs\gtx1080-eng\Lib\site-packages\torch\nn\modules\module.py", line 1527, in _call_impl
    return forward_call(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Win10\anaconda3\envs\gtx1080-eng\Lib\site-packages\torch\nn\modules\loss.py", line 618, in forward
    return F.binary_cross_entropy(input, target, weight=self.weight, reduction=self.reducti

...


Distance d=7 FAILED: Found dtype Long but expected Float

Distance d=9 (4/5)
Split: 800,000 train, 100,000 val, 100,000 test
Number of detectors: 720
SimpleNN initialized: SimpleNN(nickname='d9_specialist', in_channels=720, hidden_dims=(512, 1024, 2048), dropout=0.1)

Training with:
  hidden_dims: (512, 1024, 2048)
  learning_rate: 0.0003
  batch_size: 64
  dropout: 0.1

Training: d9_specialist
Epochs: 10 | Batch size: 64 | LR: 0.0003
Training samples: 800,000
Validation samples: 100,000




Distance d=9 FAILED: Found dtype Long but expected Float


Traceback (most recent call last):
  File "C:\Users\Win10\AppData\Local\Temp\ipykernel_18204\684687717.py", line 57, in <module>
    epoch_metrics = train_with_logging(
                    ^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Win10\AppData\Local\Temp\ipykernel_18204\3078673785.py", line 210, in train_with_logging
    loss = loss_fn(pred, y)
           ^^^^^^^^^^^^^^^^
  File "c:\Users\Win10\anaconda3\envs\gtx1080-eng\Lib\site-packages\torch\nn\modules\module.py", line 1518, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Win10\anaconda3\envs\gtx1080-eng\Lib\site-packages\torch\nn\modules\module.py", line 1527, in _call_impl
    return forward_call(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Win10\anaconda3\envs\gtx1080-eng\Lib\site-packages\torch\nn\modules\loss.py", line 618, in forward
    return F.binary_cross_entropy(input, target, weight=self.weight, reduction=self.reducti


Distance d=11 (5/5)

Distance d=11 FAILED: No baseline dataset found for d11 in ..\..\datasets\flat.
Datasets ending with '_1M' will be ignored. Please provide d11_baseline.pt or d11_baseline.json in ..\..\datasets\flat.

ALL TRAINING COMPLETE


Traceback (most recent call last):
  File "C:\Users\Win10\AppData\Local\Temp\ipykernel_18204\684687717.py", line 28, in <module>
    detections, labels = load_or_generate_dataset(d, TOTAL_SAMPLES)
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Win10\AppData\Local\Temp\ipykernel_18204\3181282205.py", line 93, in load_or_generate_dataset
    raise FileNotFoundError(
FileNotFoundError: No baseline dataset found for d11 in ..\..\datasets\flat.
Datasets ending with '_1M' will be ignored. Please provide d11_baseline.pt or d11_baseline.json in ..\..\datasets\flat.


## Results Summary

In [23]:
# =============================================================================
# LOAD AND DISPLAY RESULTS
# =============================================================================

results_path = RESULTS_DIR / "results.json"

if results_path.exists():
    with open(results_path, 'r') as f:
        results = json.load(f)

    # Filter to completed results
    completed = [r for r in results if r.get('status') == 'completed']
    print(f"Completed: {len(completed)} / {len(results)}")

    if completed:
        # Create summary DataFrame
        df_data = []
        for r in completed:
            df_data.append({
                'Distance': r['distance'],
                'Detectors': r['num_detectors'],
                'Val Acc': f"{r['val_accuracy']*100:.2f}%",
                'Test Acc': f"{r['test_accuracy']*100:.2f}%",
                'Final Loss': f"{r.get('final_loss', 0):.4f}",
                'Time (min)': f"{r.get('training_time_seconds', 0)/60:.1f}",
            })

        df = pd.DataFrame(df_data)
        df = df.sort_values('Distance')

        print("\nSimpleNN Specialist Models - Training Results:")
        print(df.to_string(index=False))

        print("\n" + "="*60)
        print("HYPERPARAMETERS USED (from tuning):")
        print("="*60)
        hp = completed[0]['hyperparams']
        print(f"  hidden_dims: {hp['hidden_dims']}")
        print(f"  learning_rate: {hp['learning_rate']}")
        print(f"  batch_size: {hp['batch_size']}")
        print(f"  dropout: {hp['dropout']}")
else:
    print("No results file found. Run the training loop first.")

Completed: 0 / 10


In [24]:
# =============================================================================
# PLOT RESULTS
# =============================================================================

if results_path.exists() and len(completed) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Sort by distance
    sorted_results = sorted(completed, key=lambda x: x['distance'])
    distances = [r['distance'] for r in sorted_results]
    test_accs = [r['test_accuracy'] * 100 for r in sorted_results]
    val_accs = [r['val_accuracy'] * 100 for r in sorted_results]

    # Plot 1: Test accuracy by distance
    ax1 = axes[0]
    ax1.bar(distances, test_accs, color='steelblue', edgecolor='black')
    ax1.set_xlabel('Code Distance (d)')
    ax1.set_ylabel('Test Accuracy (%)')
    ax1.set_title('SimpleNN Specialist Models - Test Accuracy')
    ax1.set_xticks(distances)
    for i, (d, acc) in enumerate(zip(distances, test_accs)):
        ax1.text(d, acc + 0.5, f'{acc:.1f}%', ha='center', va='bottom', fontsize=9)

    # Plot 2: Val vs Test accuracy
    ax2 = axes[1]
    x = np.arange(len(distances))
    width = 0.35
    ax2.bar(x - width/2, val_accs, width, label='Validation', color='coral', edgecolor='black')
    ax2.bar(x + width/2, test_accs, width, label='Test', color='steelblue', edgecolor='black')
    ax2.set_xlabel('Code Distance (d)')
    ax2.set_ylabel('Accuracy (%)')
    ax2.set_title('Validation vs Test Accuracy')
    ax2.set_xticks(x)
    ax2.set_xticklabels(distances)
    ax2.legend()

    plt.tight_layout()

    # Save plot
    plot_path = PLOTS_DIR / "training_results.png"
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    print(f"Plot saved to: {plot_path}")

    plt.show()

In [ ]:
from google.colab import runtime
runtime.unassign()